# microgpt.py 中文拆解版

这个 notebook 的目标不是追求训练效果，而是把 `microgpt.py` 拆成一小步一小步，让你看清楚 GPT 训练到底在做什么。

最核心的一句话：GPT 做的是 **根据前面的 token，预测下一个 token**。

训练时：模型猜下一个字符，算它猜错多少，然后根据梯度改参数。  
生成时：模型从开始符号出发，一次抽一个字符，直到抽到结束符号。


## 0. 先看全局流程

`microgpt.py` 可以分成 7 块：

1. 读入名字数据集。
2. 把字符变成整数 token。
3. 用 `Value` 实现极简自动求导。
4. 初始化 GPT 的参数矩阵。
5. 定义 GPT 前向计算：embedding、attention、MLP、logits。
6. 用 loss 和 Adam 优化器训练参数。
7. 用训练后的模型生成新名字。

下面每一节都对应原脚本的一块。建议按顺序运行 cell。

## 1. 数据集：模型要从名字里学什么？

这里的数据是一行一个名字，例如 `emma`、`olivia`。  
模型看到的训练任务不是“理解名字含义”，而是学习字符之间的统计规律：

- `e` 后面经常接什么？
- `em` 后面经常接什么？
- 一个名字通常什么时候结束？


In [ ]:
from pathlib import Path
import math
import random

random.seed(42)

input_path = Path('input.txt')
if not input_path.exists():
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, input_path)

docs = [line.strip() for line in input_path.open() if line.strip()]
random.shuffle(docs)

print('样本数量:', len(docs))
print('前 10 个名字:', docs[:10])
print('最长名字长度:', max(len(d) for d in docs))


## 2. Tokenizer：把字符变成数字

神经网络不能直接处理字符串，所以要先把字符变成整数。这个脚本采用最简单的 **字符级 tokenizer**：

- 每个不同字符分配一个整数 id。
- 额外增加一个特殊 token：`BOS`。
- 在这个脚本里，`BOS` 同时表示开始和结束。

如果名字是 `emma`，训练序列可以写成：

$$x = [BOS, e, m, m, a, BOS]$$

模型在第 `t` 个位置看到 `x_t`，目标是预测下一个 token：

$$target_t = x_{t+1}$$

In [ ]:
uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

stoi = {ch: i for i, ch in enumerate(uchars)}
itos = {i: ch for ch, i in stoi.items()}
itos[BOS] = '<BOS>'

def encode(text):
    return [stoi[ch] for ch in text]

def decode(token_ids):
    return ''.join(itos[i] for i in token_ids if i != BOS)

def token_name(token_id):
    return '<BOS>' if token_id == BOS else uchars[token_id]

doc = 'emma' if 'emma' in docs else docs[0]
tokens = [BOS] + encode(doc) + [BOS]

print('词表字符:', uchars)
print('词表大小:', vocab_size)
print('示例名字:', doc)
print('token 序列:', tokens)
print('token 含义:', [token_name(t) for t in tokens])
print()
for i, tok in enumerate(tokens[:-1]):
    print(f'位置 {i:2d}: 输入 {token_name(tok):>5} -> 目标 {token_name(tokens[i + 1])}')


## 3. 自动求导：为什么模型知道该怎么改参数？

`Value` 是这个脚本最关键的教学组件之一。它表示一个标量，同时记住：

- 自己的数值 `data`。
- 最终 loss 对自己的导数 `grad`。
- 自己由哪些子节点算出来。
- 自己对每个子节点的局部导数。

反向传播就是反复使用链式法则：

$$dL/dx = dL/dy * dy/dx$$

直觉：如果 `x` 影响 `y`，而 `y` 又影响 `L`，那么 `x` 对 `L` 的影响就是这两段影响乘起来。

In [ ]:
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other):
        return Value(self.data**other, (self,), (other * self.data**(other - 1),))

    def log(self):
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def exp(self):
        return Value(math.exp(self.data), (self,), (math.exp(self.data),))

    def relu(self):
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad


# 一个很小的例子：loss = (a * b + c)^2
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
d = a * b + c
loss = d**2
loss.backward()

print('d =', d.data)
print('loss =', loss.data)
print('d(loss)/d(a) =', a.grad)
print('d(loss)/d(b) =', b.grad)
print('d(loss)/d(c) =', c.grad)


## 4. 三个基础函数：linear、softmax、rmsnorm

GPT 前向计算里会反复出现这几个基础操作。

### Linear

把输入向量 `x` 乘以权重矩阵 `W`：

$$y_i = sum_j W_{ij} x_j$$

### Softmax

把一组任意实数打分 `z` 变成概率：

$$p_i = exp(z_i) / sum_j exp(z_j)$$

### RMSNorm

把向量缩放到比较稳定的范围：

$$RMS(x) = sqrt(mean(x_i^2))$$

$$RMSNorm(x_i) = x_i / sqrt(mean(x_i^2) + epsilon)$$

In [ ]:
def linear(x, w):
    # w 的每一行负责算输出向量的一个元素
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax(logits):
    # 减最大值是数值稳定技巧，不改变 softmax 的结果比例
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

# 看一个 softmax 小例子
logits = [Value(1.0), Value(2.0), Value(0.5)]
probs = softmax(logits)
print('概率:', [round(p.data, 4) for p in probs])
print('概率之和:', round(sum(p.data for p in probs), 4))


## 5. 注意力：当前字符应该看前面哪些字符？

自注意力的作用：当前 token 根据上下文，决定应该关注前面哪些 token。

每个位置会生成三种向量：

- `q` / query：我现在想找什么信息？
- `k` / key：我这个位置有什么特征？
- `v` / value：如果别人关注我，真正拿走的信息是什么？

注意力分数：

$$score_t = q dot k_t / sqrt(d)$$

注意力权重：

$$a = softmax(score)$$

最后输出是所有 value 的加权平均：

$$out = sum_t a_t v_t$$

In [ ]:
def softmax_float(xs):
    max_x = max(xs)
    exps = [math.exp(x - max_x) for x in xs]
    total = sum(exps)
    return [x / total for x in exps]

# 这个例子不用 Value，只是为了直观看注意力权重怎么来
q = [1.0, 0.0]
keys_toy = [
    [1.0, 0.0],  # 和 q 很像，应该被多关注
    [0.0, 1.0],  # 和 q 不像
    [0.7, 0.7],  # 有点像
]
values_toy = [
    [10.0, 0.0],
    [0.0, 10.0],
    [5.0, 5.0],
]

scores = [sum(q[j] * k[j] for j in range(2)) / math.sqrt(2) for k in keys_toy]
weights = softmax_float(scores)
out = [sum(weights[t] * values_toy[t][j] for t in range(3)) for j in range(2)]

print('attention scores:', [round(x, 4) for x in scores])
print('attention weights:', [round(x, 4) for x in weights])
print('weighted value output:', [round(x, 4) for x in out])


## 6. 参数：模型的“知识”存在一堆数字里

训练 GPT，本质就是调整很多权重数字。这里的参数包括：

- `wte`：token embedding，表示“这个字符是谁”。
- `wpe`：position embedding，表示“这个字符在第几个位置”。
- `attn_wq / attn_wk / attn_wv / attn_wo`：注意力层参数。
- `mlp_fc1 / mlp_fc2`：前馈网络参数。
- `lm_head`：把模型内部向量变回词表 logits 的参数。

embedding 的第一步可以理解成：

$$x = token\_embedding[token\_id] + position\_embedding[pos\_id]$$

In [ ]:
n_layer = 1
n_embd = 16
block_size = 16
n_head = 4
head_dim = n_embd // n_head

def matrix(nout, nin, std=0.08):
    return [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]

def reset_model(seed=42):
    random.seed(seed)
    state_dict = {
        'wte': matrix(vocab_size, n_embd),
        'wpe': matrix(block_size, n_embd),
        'lm_head': matrix(vocab_size, n_embd),
    }
    for i in range(n_layer):
        state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
        state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
        state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
        state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
        state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
        state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
    params = [p for mat in state_dict.values() for row in mat for p in row]
    return state_dict, params

state_dict, params = reset_model()

print('层数 n_layer:', n_layer)
print('向量宽度 n_embd:', n_embd)
print('注意力头数 n_head:', n_head)
print('每个头的宽度 head_dim:', head_dim)
print('参数总数:', len(params))
print('wte 形状:', len(state_dict['wte']), 'x', len(state_dict['wte'][0]))


## 7. GPT 前向传播：从一个 token 到下一个 token 的打分

`gpt(token_id, pos_id, keys, values)` 做的事情：

1. 取当前 token embedding 和 position embedding。
2. 进入 Transformer layer。
3. 先做多头注意力，融合前文信息。
4. 再做 MLP，增加非线性表达能力。
5. 最后用 `lm_head` 得到词表中每个 token 的 logits。

logits 还不是概率。要经过 softmax 才能变成概率：

$$prob = softmax(logits)$$

In [ ]:
def gpt(token_id, pos_id, keys, values):
    # 1. 当前字符是谁 + 当前在第几个位置
    tok_emb = state_dict['wte'][token_id]
    pos_emb = state_dict['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]
    x = rmsnorm(x)

    for li in range(n_layer):
        # 2. 多头自注意力
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])

        keys[li].append(k)
        values[li].append(v)

        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs + head_dim]
            k_h = [ki[hs:hs + head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs + head_dim] for vi in values[li]]

            attn_logits = [
                sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5
                for t in range(len(k_h))
            ]
            attn_weights = softmax(attn_logits)
            head_out = [
                sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h)))
                for j in range(head_dim)
            ]
            x_attn.extend(head_out)

        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]

        # 3. MLP 前馈网络
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    # 4. 输出每个 token 作为下一个 token 的打分
    logits = linear(x, state_dict['lm_head'])
    return logits


## 8. Loss：怎么衡量“猜错了多少”？

模型输出 logits，softmax 后得到每个 token 的概率。假设正确答案是 `target`，交叉熵 loss 是：

$$loss = -log(prob[target])$$

所以：

- 正确 token 概率越高，loss 越低。
- 正确 token 概率越低，loss 越高。

一个名字有多个位置，例如 `[BOS, e, m, m, a, BOS]` 有 5 次预测。最终 loss 是这些位置 loss 的平均值。

In [ ]:
def top_k_tokens(probs, k=5):
    ranked = sorted(range(len(probs)), key=lambda i: probs[i].data, reverse=True)
    return [(token_name(i), probs[i].data) for i in ranked[:k]]

def forward_doc(doc, show=True):
    tokens = [BOS] + encode(doc) + [BOS]
    n = min(block_size, len(tokens) - 1)
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]
    losses = []

    for pos_id in range(n):
        token_id = tokens[pos_id]
        target_id = tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)

        if show:
            best = top_k_tokens(probs, k=3)
            best_text = ', '.join(f'{tok}:{prob:.3f}' for tok, prob in best)
            print(f'位置 {pos_id:2d} | 输入 {token_name(token_id):>5} | 正确答案 {token_name(target_id):>5} | top3 {best_text}')

    return (1 / n) * sum(losses)

demo_doc = 'emma' if 'emma' in docs else docs[0]
loss = forward_doc(demo_doc, show=True)
print('\n当前平均 loss:', round(loss.data, 4))

# 反向传播后，每个参数都会得到一个 grad，表示这个参数该怎么调整才能降低 loss。
loss.backward()
print('第一个参数的梯度示例:', params[0].grad)

# 清空刚才演示产生的梯度，避免影响下一节训练。
for p in params:
    p.grad = 0


## 9. Adam：根据梯度更新参数

最简单的梯度下降是：

$$p = p - lr * grad$$

Adam 会更稳一点，因为它维护两组历史统计量：

$$m_t = beta_1 * m_{t-1} + (1 - beta_1) * grad$$

$$v_t = beta_2 * v_{t-1} + (1 - beta_2) * grad^2$$

直觉上：

- `m` 像是梯度方向的移动平均。
- `v` 像是梯度大小的移动平均。
- 这样参数更新不会完全被某一步的偶然大梯度带偏。

In [ ]:
learning_rate = 0.01
beta1 = 0.85
beta2 = 0.99
eps_adam = 1e-8

m = [0.0] * len(params)
v = [0.0] * len(params)

def train(num_steps=80):
    history = []
    for step in range(num_steps):
        doc = docs[step % len(docs)]
        loss = forward_doc(doc, show=False)
        loss.backward()

        lr_t = learning_rate * (1 - step / num_steps)
        for i, p in enumerate(params):
            m[i] = beta1 * m[i] + (1 - beta1) * p.grad
            v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
            m_hat = m[i] / (1 - beta1 ** (step + 1))
            v_hat = v[i] / (1 - beta2 ** (step + 1))
            p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
            p.grad = 0

        history.append(loss.data)
        if step % 10 == 0 or step == num_steps - 1:
            print(f'step {step + 1:3d}/{num_steps} | loss {loss.data:.4f}')
    return history

# 这里只训练 80 步，目的是看懂流程，不是追求好效果。
history = train(num_steps=80)


## 10. 生成：让模型一个字符一个字符地写名字

生成过程和训练时的前向传播类似，但没有 target，也不会 backward。

流程：

1. 从 `BOS` 开始。
2. 模型给出下一个 token 的概率。
3. 按概率随机抽一个 token。
4. 把抽到的 token 当作下一步输入。
5. 如果抽到 `BOS`，表示名字结束。

`temperature` 控制随机性：

- 小：更保守，更容易选高概率字符。
- 大：更随机，更容易产生奇怪结果。

In [ ]:
def generate_one(temperature=0.7):
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []

    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])

    return ''.join(sample)

for i in range(10):
    print(f'sample {i + 1:2d}:', generate_one(temperature=0.7))


## 11. 和 `microgpt.py` 的对应关系

| notebook 小节 | 对应原脚本内容 | 你应该抓住的重点 |
| --- | --- | --- |
| 数据集 | 读取 `input.txt` | 训练材料是一堆名字 |
| Tokenizer | `uchars`、`BOS`、`vocab_size` | 字符被映射成整数 |
| 自动求导 | `class Value` | loss 可以一路反传到每个参数 |
| 参数初始化 | `state_dict`、`params` | 模型知识存在权重数字里 |
| GPT 前向 | `gpt(...)` | 输入当前 token，输出下一个 token 的 logits |
| Loss | `-probs[target_id].log()` | 正确答案概率越低，惩罚越大 |
| Adam | `m`、`v`、`p.data -= ...` | 用梯度一点点调整参数 |
| 生成 | 最后的 inference 循环 | 没有答案，只按概率抽下一个字符 |

如果你第一次看 GPT，建议先不用纠结每个矩阵乘法的细节。先记住这个主线：

**token -> embedding -> attention/MLP -> logits -> softmax -> loss -> backward -> update**

## 12. 可以自己改的几个实验

你可以尝试改下面这些变量，然后重新运行相关 cell：

- `num_steps`：训练更久，生成结果通常更像名字。
- `temperature`：调高更随机，调低更保守。
- `n_embd`：向量更宽，模型容量更大，但计算更慢。
- `n_head`：注意力头更多，可以学习更多种关注模式。
- `demo_doc`：换一个名字，看每个位置的预测 top3。